<a href="https://colab.research.google.com/github/Tharindusam99/PassPro/blob/Defensive-Skills-Analyser/NetBall_Defence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install mediapipe opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 47.3 MB/s eta 0:00:00


In [ ]:

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

In [ ]:

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

Processing frame 10 of 4892 (0.2%)
Processing frame 20 of 4892 (0.4%)
Processing frame 30 of 4892 (0.6%)
Processing frame 40 of 4892 (0.8%)
Processing frame 50 of 4892 (1.0%)
Processing frame 60 of 4892 (1.2%)
Processing frame 70 of 4892 (1.4%)
Processing frame 80 of 4892 (1.6%)
Processing frame 90 of 4892 (1.8%)
Processing frame 100 of 4892 (2.0%)
Processing frame 110 of 4892 (2.2%)
Processing frame 120 of 4892 (2.5%)
Processing frame 130 of 4892 (2.7%)
Processing frame 140 of 4892 (2.9%)
Processing frame 150 of 4892 (3.1%)
Processing frame 160 of 4892 (3.3%)
Processing frame 170 of 4892 (3.5%)
Processing frame 180 of 4892 (3.7%)
Processing frame 190 of 4892 (3.9%)
Processing frame 200 of 4892 (4.1%)
Processing frame 210 of 4892 (4.3%)
Processing frame 220 of 4892 (4.5%)
Processing frame 230 of 4892 (4.7%)
Error in frame 236: 'NoneType' object has no attribute 'landmark'
Processing frame 240 of 4892 (4.9%)
Processing frame 250 of 4892 (5.1%)
Processing frame 260 of 4892 (5.3%)
Process

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from moviepy.editor import ImageSequenceClip, VideoFileClip, clips_array
import os

def calculate_angle(a, b, c):
    """Calculate angle between three points"""
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)

    if angle > 180.0:
        angle = 360-angle

    return angle

def calculate_hip_knee_distance(left_hip, right_hip, left_knee, right_knee):
    """Calculate the average distance between hips and knees for stance width"""
    left_distance = np.sqrt((left_hip[0] - left_knee[0])**2 + (left_hip[1] - left_knee[1])**2)
    right_distance = np.sqrt((right_hip[0] - right_knee[0])**2 + (right_hip[1] - right_knee[1])**2)
    return (left_distance + right_distance) / 2

def process_frame(frame, pose, mp_pose, mp_drawing):
    """Process a single frame and return angle measurements"""
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False

    results = pose.process(image)

    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    angles = {}

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark

        # Get coordinates for defensive stance analysis
        left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,
                   landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
        right_hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x,
                    landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
        left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
                    landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
        right_knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x,
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
        left_ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x,
                     landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]
        right_ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x,
                      landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]

        # Calculate key angles for defensive stance
        angles['left_knee'] = calculate_angle(left_hip, left_knee, left_ankle)
        angles['right_knee'] = calculate_angle(right_hip, right_knee, right_ankle)
        angles['hip_stance'] = calculate_angle(left_hip,
                                            [(left_hip[0] + right_hip[0])/2,
                                             (left_hip[1] + right_hip[1])/2],
                                            right_hip)

        # Calculate stance width (distance between hips and knees)
        angles['stance_width'] = calculate_hip_knee_distance(left_hip, right_hip, left_knee, right_knee)

        # Draw angles on frame
        h, w = frame.shape[:2]
        cv2.putText(image, f"Left knee: {angles['left_knee']:.1f}°",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(image, f"Right knee: {angles['right_knee']:.1f}°",
                    (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(image, f"Hip stance: {angles['hip_stance']:.1f}°",
                    (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(image, f"Stance width: {angles['stance_width']:.3f}",
                    (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        # Draw pose landmarks
        mp_drawing.draw_landmarks(
            image,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
        )

    return image, angles

def extract_angle_data(angles_list):
    """Extract separate lists for each angle type from the angles data"""
    left_knee_angles = [frame['left_knee'] for frame in angles_list]
    right_knee_angles = [frame['right_knee'] for frame in angles_list]
    hip_stance_angles = [frame['hip_stance'] for frame in angles_list]
    stance_widths = [frame['stance_width'] for frame in angles_list]

    return left_knee_angles, right_knee_angles, hip_stance_angles, stance_widths

def create_angle_animation(correct_angles, incorrect_angles, fps):
    """Creates an animated graph comparing defense stance metrics over time"""
    if not os.path.exists('temp_frames'):
        os.makedirs('temp_frames')

    # Extract individual angle data
    correct_lk, correct_rk, correct_hip, correct_width = extract_angle_data(correct_angles)
    incorrect_lk, incorrect_rk, incorrect_hip, incorrect_width = extract_angle_data(incorrect_angles)

    max_frames = max(len(correct_angles), len(incorrect_angles))
    times = list(range(max_frames))

    graph_frames = []
    for frame in range(max_frames):
        fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 16))

        # Plot knee angles
        if frame < len(correct_lk):
            ax1.plot(times[:frame+1], correct_lk[:frame+1],
                    'g-', linewidth=2, label='Correct Left Knee')
            ax1.plot(times[:frame+1], correct_rk[:frame+1],
                    'b-', linewidth=2, label='Correct Right Knee')
        if frame < len(incorrect_lk):
            ax1.plot(times[:frame+1], incorrect_lk[:frame+1],
                    'r-', linewidth=2, label='Incorrect Left Knee')
            ax1.plot(times[:frame+1], incorrect_rk[:frame+1],
                    'm-', linewidth=2, label='Incorrect Right Knee')
        ax1.set_ylabel('Knee Angles (degrees)')
        ax1.set_title('Knee Bend Comparison')
        ax1.grid(True)
        ax1.legend()

        # Plot hip stance
        if frame < len(correct_hip):
            ax2.plot(times[:frame+1], correct_hip[:frame+1],
                    'g-', linewidth=2, label='Correct Technique')
        if frame < len(incorrect_hip):
            ax2.plot(times[:frame+1], incorrect_hip[:frame+1],
                    'r-', linewidth=2, label='Incorrect Technique')
        ax2.set_ylabel('Hip Stance Angle (degrees)')
        ax2.set_title('Hip Stance Comparison')
        ax2.grid(True)
        ax2.legend()

        # Plot stance width
        if frame < len(correct_width):
            ax3.plot(times[:frame+1], correct_width[:frame+1],
                    'g-', linewidth=2, label='Correct Technique')
        if frame < len(incorrect_width):
            ax3.plot(times[:frame+1], incorrect_width[:frame+1],
                    'r-', linewidth=2, label='Incorrect Technique')
        ax3.set_ylabel('Stance Width')
        ax3.set_xlabel('Frame Number')
        ax3.set_title('Defensive Stance Width Comparison')
        ax3.grid(True)
        ax3.legend()

        # Add overall stance quality metric
        if frame < len(correct_width):
            correct_quality = [(90 - abs(90 - k))/90 * 100 for k in correct_lk[:frame+1]]
            ax4.plot(times[:frame+1], correct_quality,
                    'g-', linewidth=2, label='Correct Form')
        if frame < len(incorrect_width):
            incorrect_quality = [(90 - abs(90 - k))/90 * 100 for k in incorrect_lk[:frame+1]]
            ax4.plot(times[:frame+1], incorrect_quality,
                    'r-', linewidth=2, label='Incorrect Form')
        ax4.set_ylabel('Stance Quality (%)')
        ax4.set_xlabel('Frame Number')
        ax4.set_title('Overall Defensive Stance Quality')
        ax4.grid(True)
        ax4.legend()

        plt.tight_layout()
        frame_path = f'temp_frames/graph_{frame:04d}.png'
        plt.savefig(frame_path)
        graph_frames.append(frame_path)
        plt.close()

    clip = ImageSequenceClip(graph_frames, fps=fps)
    return clip, graph_frames

def analyze_defensive_movement(correct_video_path, incorrect_video_path, output_path):
    """
    Complete analysis pipeline for defensive movement comparison

    Parameters:
    correct_video_path (str): Path to video with correct defensive technique
    incorrect_video_path (str): Path to video with incorrect defensive technique
    output_path (str): Path where the final analysis video will be saved
    """
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils

    cap1 = cv2.VideoCapture(correct_video_path)
    cap2 = cv2.VideoCapture(incorrect_video_path)

    width = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap1.get(cv2.CAP_PROP_FPS))

    if not os.path.exists('temp_video_frames'):
        os.makedirs('temp_video_frames')

    correct_angles = []
    incorrect_angles = []
    video_frames = []
    frame_count = 0

    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        while True:
            ret1, frame1 = cap1.read()
            ret2, frame2 = cap2.read()

            if not ret1 or not ret2:
                break

            processed1, angles1 = process_frame(frame1, pose, mp_pose, mp_drawing)
            processed2, angles2 = process_frame(frame2, pose, mp_pose, mp_drawing)

            if angles1 and angles2:
                correct_angles.append(angles1)
                incorrect_angles.append(angles2)

            # Add labels to distinguish correct vs incorrect technique
            cv2.putText(processed1, "Correct Technique", (10, height - 20),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(processed2, "Incorrect Technique", (10, height - 20),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            combined_frame = np.hstack((processed1, processed2))

            frame_path = f'temp_video_frames/frame_{frame_count:04d}.png'
            cv2.imwrite(frame_path, combined_frame)
            video_frames.append(frame_path)
            frame_count += 1

    cap1.release()
    cap2.release()

    video_clip = ImageSequenceClip(video_frames, fps=fps)
    graph_clip, graph_frames = create_angle_animation(correct_angles, incorrect_angles, fps)

    graph_clip = graph_clip.resize(height=video_clip.h)
    final_clip = clips_array([[video_clip, graph_clip]])

    final_clip.write_videofile(output_path, codec='libx264')

    # Clean up temporary files
    for frame in video_frames:
        os.remove(frame)
    for frame in graph_frames:
        os.remove(frame)
    os.rmdir('temp_video_frames')
    os.rmdir('temp_frames')

    video_clip.close()
    graph_clip.close()
    final_clip.close()

if __name__ == "__main__":
    correct_video_path = "/content/drive/MyDrive/NetballProject/Netball vidoes/Defense/IMG_7678.MOV"
    incorrect_video_path = "/content/drive/MyDrive/NetballProject/Netball vidoes/Defense/Wrong/IMG_7692.MOV"
    output_path = "/content/drive/MyDrive/NetballProject/defense_movement_analysis.mp4"

    analyze_defensive_movement(correct_video_path, incorrect_video_path, output_path)

  if event.key is 'enter':



Moviepy - Building video /content/drive/MyDrive/NetballProject/defense_movement_analysis.mp4.
Moviepy - Writing video /content/drive/MyDrive/NetballProject/defense_movement_analysis.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/NetballProject/defense_movement_analysis.mp4
